In [1]:
!pip install PyMuPDF sentence-transformers qdrant-client[http] --upgrade


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import fitz  # PyMuPDF
import os

# === Provide the full path to your PDF file ===
pdf_path = os.path.expanduser("~/Downloads/Andrei Gheorghiu - Building Data-Driven Applications with LlamaIndex_ A practical guide to retrieval-a.pdf")

if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found at: {pdf_path}")

print("✅ PDF found:", pdf_path)


✅ PDF found: /home/sohan/Downloads/Andrei Gheorghiu - Building Data-Driven Applications with LlamaIndex_ A practical guide to retrieval-a.pdf


In [3]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc): #i for page number starting from 0 and so on and for page it stores the containt of that page
   
        
        text = page.get_text("text") #function to extratract text from the current page and argugument "text" tells strictly to extract only text
        pages.append({"page": i + 1, "text": text}) # append dictionary in the list pages with page number and 
    return pages

pages = extract_text_from_pdf(pdf_path)
print(f"Extracted {len(pages)} pages.")




Extracted 367 pages.


In [4]:
from qdrant_client import QdrantClient
from qdrant_client.http import models as rest

QDRANT_URL = "http://localhost:6333" #Address of qudrant databases
COLLECTION_NAME = "pdf_rag_col_v1" #name of database to store vectors

qdrant = QdrantClient(url=QDRANT_URL) #for making connection which is running locally
print("✅ Connected to Qdrant at:", QDRANT_URL)

# Check if collection exists
collections = [c.name for c in qdrant.get_collections().collections]
print("Existing collections:", collections)


✅ Connected to Qdrant at: http://localhost:6333
Existing collections: []


In [5]:
import uuid#provides unique identity we will use this to provide unique identity for each chunk

#Its purpose is to break down the large blocks of text from each page into smaller, 
#well-defined pieces (chunks) that a language model can effectively work with.
#The "overlap" is like re-reading the last paragraph of the previous 
#section before you start a new one, just to remember the context and keep the flow.

def chunk_text(pages, chunk_size=1000, chunk_overlap=200): #1000 no of charceters in each chunk and 
                                                           #200 is number of character which next will share
    """
    Splits extracted PDF pages into overlapping text chunks.
    This ensures the context is preserved between chunks.
    """
    chunks = [] #holds all final chunks dictionaries
    for p in pages:
        text = p["text"].strip() #.strip() to remove any accidental blank spaces from the beginning or end and text is full containt of page
        page_num = p["page"]

        if not text: #to check if page only contains images or it is blank
            continue

        start = 0 #starting of text in each page
        idx = 0 # Initializes a counter for the chunks on this specific page.
        while start < len(text):#
            end = start + chunk_size
            chunk = text[start:end].strip()
            if chunk: #checks if chunk is not empty
                chunks.append({
                    "id": str(uuid.uuid4()), #unique id
                    "page": page_num, #page number
                    "chunk_index": idx, #Sequential number chunk within the page
                    "text": chunk
                })
                idx += 1
            start = end - chunk_overlap  #overlap for context
    return chunks

print("✅ Step 5A: Chunking function ready.")


✅ Step 5A: Chunking function ready.


In [6]:
from sentence_transformers import SentenceTransformer #This library is specifically designed to make it easy to 
                                                      #download, load, and use state-of-the-art embedding models.
                                                     #SentenceTransformer GEnrates embeddings for each sentence
embed_model_name = "all-MiniLM-L6-v2" # Embedding model
embed_model = SentenceTransformer(embed_model_name) #The code tells the SentenceTransformer library to load the model we specified.
                                                   #The embed_model variable now holds a ready-to-use 
                                                  #object capable of performing the text-to-vector "translation."
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print(f"✅ Step 5B: Loaded embedding model '{embed_model_name}' with dimension {EMBED_DIM}.")


✅ Step 5B: Loaded embedding model 'all-MiniLM-L6-v2' with dimension 384.


In [7]:
from qdrant_client.http import models as rest

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    print(f"Collection '{COLLECTION_NAME}' already exists.")
else:
    print(f"Creating collection '{COLLECTION_NAME}' ...")
    qdrant.recreate_collection( #Command sent to qudrant that Create a new, empty collection with the following specifications
        collection_name=COLLECTION_NAME,
        vectors_config=rest.VectorParams(size=EMBED_DIM, distance=rest.Distance.COSINE) #Its the blueprint for the vector we store
                                                            #and vector of specific dimension    #To use cosine similarities
    )

print("✅ Step 5C: Collection is ready in Qdrant.")


Creating collection 'pdf_rag_col_v1' ...
✅ Step 5C: Collection is ready in Qdrant.


/tmp/ipykernel_3887/860225473.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection( #Command sent to qudrant that Create a new, empty collection with the following specifications


In [8]:
chunks = chunk_text(pages)
print(f"✅ Step 5D: Created {len(chunks)} text chunks from the PDF.")

def batch_generator(data, batch_size=64):
    for i in range(0, len(data), batch_size): # This slices the main list, grabbing a small batch  
                                                #(e.g., items from 0 to 63, then 64 to 127, etc.).
        yield data[i:i + batch_size] # it produces one batch, pauses, gives it to the code that called it, 
                                    #and then waits until it's asked for the next one

print("Batch generator ready for embedding.")


✅ Step 5D: Created 1097 text chunks from the PDF.
Batch generator ready for embedding.


In [9]:
'''This block of code is the engine of the entire indexing process. 
It takes all the text chunks we prepared, converts them into numerical vectors using the AI model, 
and then uploads everything to your Qdrant database'''

from qdrant_client.http import models as rest
import numpy as np

print("🚀 Step 5E: Starting embedding + upsert process...")

for batch in batch_generator(chunks, batch_size=64):
    texts = [c["text"] for c in batch] #Extract Texts for the Batch
    embeddings = embed_model.encode(texts, convert_to_numpy=True) #crates embeddings

    points = []
    for c, vector in zip(batch, embeddings):
        points.append(
            rest.PointStruct(
                id=c["id"],
                vector=vector.tolist(),
                payload={  #extra data
                    "page": c["page"],
                    "chunk_index": c["chunk_index"],
                    "text": c["text"]
                }
            )
        )

    qdrant.upsert(collection_name=COLLECTION_NAME, points=points) #Inserting vectors into qdrant database

print("✅ Step 5E: All chunks embedded and stored in Qdrant successfully.")


🚀 Step 5E: Starting embedding + upsert process...
✅ Step 5E: All chunks embedded and stored in Qdrant successfully.


In [10]:
count = qdrant.count(collection_name=COLLECTION_NAME).count #For counting of vectors
print(f"Total stored vectors: {count}")


Total stored vectors: 1097


In [11]:
import numpy as np
#Retrieval Step
def search_qdrant(query: str, top_k: int = 5):
    """
    Searches Qdrant for top_k chunks relevant to the query.
    Returns a list of hits with text and scores.
    """
    query_vector = embed_model.encode([query], convert_to_numpy=True)[0].astype(np.float32)
    hits = qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector.tolist(),
        limit=top_k
    )
    return hits

print("✅ Step 6A: Qdrant search function ready.")


✅ Step 6A: Qdrant search function ready.


In [12]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1:7b"

def ask_deepseek(query: str, top_k: int = 5):
    """
    Uses retrieved context from Qdrant + local DeepSeek model to answer a query.
    """
    # 1. Retrieve top chunks from Qdrant
    hits = search_qdrant(query, top_k=top_k)
    if not hits:
        return "No relevant context found."

    # 2. Combine retrieved text into one context block
    context = "\n\n".join([h.payload["text"] for h in hits])
    prompt = f"""You are a helpful assistant. Use the following context from a PDF to answer the question.

Context:
{context}

Question:
{query}

Answer in a clear and concise way.
"""

    # 3. Send to local DeepSeek model via Ollama
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    if response.status_code != 200:
        raise Exception(f"Ollama request failed: {response.text}")

    result = response.json()
    answer = result.get("response", "").strip()
    return answer

print("✅ Step 6B: DeepSeek query function ready.")


✅ Step 6B: DeepSeek query function ready.


In [13]:
query = input("Ask a question about your PDF: ")

answer = ask_deepseek(query, top_k=5)

print("\n🧠 DeepSeek Response:\n")
print(answer)

Ask a question about your PDF:  What is RAG?


/tmp/ipykernel_3815/2095254344.py:9: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = qdrant.search(



🧠 DeepSeek Response:

<think>
Okay, so I need to figure out what RAG stands for based on the provided context. The question is asking me to define RAG, specifically from the given PDF excerpt.

Looking at the context, it's talking about an RAG application and improvements made to the RAG model. The acronym RAG isn't explicitly defined in a traditional sense like in some other contexts where you might see something like RNN or CNN. So I'll have to infer what RAG stands for based on the surrounding information.

The paragraph mentions "RAGAS enables fine-grained analysis of RAG pipelines by providing tools to visualize and interpret..." Hmm, wait, that's actually part of a different term because it starts with RAGAS. Maybe RAG refers to something else here. Wait, perhaps there was a typo or the user included an extra 's'?

Looking again: "RAG model" is mentioned, so maybe RAG stands for Retrieval-Augmented Generation? That makes sense in the context of combining retrieval and generation